In [1]:
import pandas as pd
import numpy as np
import os
%matplotlib widget
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8})
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
pd.options.display.max_seq_items = 2000
# from tqdm import tqdm
from datetime import timedelta
import re
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata, read_in_routine

In [2]:
def convert_units(dose, unit, body_weight_kg=None):
        # normalize the unit to mg (bolus) or mg/hr (infusion)
    unit = unit.lower()
    
    if unit in ['mg', 'mg/hr']:
        pass
    elif unit in ['g', 'g/hr']:
        dose = dose * 1000.
    elif unit in ['mcg', 'mcg/hr']:
        dose = dose / 1000.
    elif unit == 'mg/min':
        dose = dose * 60.
    elif unit == 'mcg/min':
        dose = dose * 60. / 1000.
    elif unit == 'mcg/hr':
        dose = dose / 1000.
    elif unit == 'mcg/h':
        dose = dose / 1000.
    elif unit == 'mcg/kg/min':
        dose = dose * 60. / 1000.* body_weight_kg
    elif unit == 'mcg/kg/hr':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mcg/kg':
        dose = dose / 1000.* body_weight_kg
    elif unit == 'mg/kg':
        dose = dose * body_weight_kg
    elif unit == 'g/kg':
        dose = dose * 1000.* body_weight_kg    
    elif unit == 'ml':
        # this is for 'infiltration' and 'nebulization'. does not happen often. this conversino just serves as an estimate (w/o density), results of those meds should be interpreted accordingly.
        dose = dose * 1000
    else:
        raise ValueError('Unknown unit: %s'%(unit))
        
    return dose


def extract_unit_from_med_name(df_med):
    
    unit = None
    med_name = pd.unique(df_med.MedicationDSC)
    assert len(med_name) == 1
    med_name = med_name[0].lower()
    
    possible_units = ['mcg', 'mg','g']
    for candidate in possible_units:
        if (' '+candidate+' ' in med_name) | (' '+candidate+'/' in med_name):
            unit = candidate
            break
            
    return unit

In [3]:
# SET PATHS FOR THE FILES NEEDED:

project = 'icu-sleep'
# project = 'sample'

if project == 'icu-sleep':
    cohort_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/LIST.csv'
    save_path_meds_filtered_for_icu_sleep_encounter = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_meds_encounter.csv'
    medication_categories_path = 'medication_categories.xlsx'
    edw_data_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_meds.csv'
    adt_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_adt.csv'
    weight_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/medications_all_timeseries'

elif project == 'sample':
    cohort_path = None
    save_path_meds_filtered_for_icu_sleep_encounter = None
    medication_categories_path = 'med_dd.xlsx'
    edw_data_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_meds.csv'
    adt_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_adt.csv'
    weight_path = '/media/mad3/Projects/covid_data/EDW/edw_covid_sample_flowsheets.csv'
    # directory where the subject-wise timeseries shall be saved:
    meds_all_output_dir = '/media/mad3/Projects/covid_data/EDW/medications_all_timeseries'

if not os.path.exists(meds_all_output_dir):
    os.makedirs(meds_all_output_dir)

In [4]:
if cohort_path is not None:
    cohort = pd.read_csv(cohort_path)
    cohort.dropna(subset=['MRN:'], inplace=True)
    cohort.loc[cohort['Study ID:']=='31', 'MRN:'] = 1876849
    cohort.loc[cohort['Study ID:']=='15', 'MRN:'] = 1479357
    cohort = cohort[cohort['Study ID:']!='98a']
    cohort['study_id'] = cohort['Study ID:'].apply(lambda x: str(x).zfill(3))
    cohort['mrn'] = cohort['MRN:'].apply(lambda x: int(x))
    cohort['datetime'] = pd.to_datetime(cohort['First Day Enrolled in Study:'], infer_datetime_format=1)
    cohort.head(2)
else:
    adt_data = pd.read_csv(adt_path)
    cohort = pd.DataFrame([])
    cohort['mrn'] = adt_data.MRN.unique()
    cohort['datetime'] = ['2020-04-08', '2020-04-04']
    cohort['datetime'] = pd.to_datetime(cohort['datetime'], infer_datetime_format=1)

In [5]:
med_categories = pd.read_excel(medication_categories_path)
med_categories.columns = [x.replace('-','_').replace('\n','').replace(': ','_') for x in med_categories.columns]
for col in med_categories.columns:
    med_categories[col] = med_categories[col].str.lower()
med_categories.head(3)



,vasopressor_Sympathomimetics,vasopressor_S_alkylisothiouronium,vasopressor_gluco/mineralocorticoids,vasopressor_analeptics,vasopressor_Pyschotropics,vasopressor_Cardiac glycosides,vasopressor_Other,antihypertensive_Thiazide_diuretics,antihypertensive_Loop_diuretics,antihypertensive_K_sparing diuretics,antihypertensive_Dihydropyridine Ca_blockers,antihypertensive_Non_Dihydropyridine Ca_blockers,antihypertensive_ACE_Inhibitors,antihypertensive_ARBs,antihypertensive_Beta Blockers,antihypertensive_Alpha_Blockers,antihypertensive_Mixed_Blockers,Antihypertensive_Alpha_2 agonists,Antihypertensive_Other,Sedative_Analgesic_Analgesic_Opioids,Sedative_Analgesic_Antidepressants,Sedative_Analgesic_Antipsychotics,Sedative_Analgesic_Benzodiazepines,Sedative_Analgesic_Antihistamines,Sedative_Analgesic_General Anesthetics,Sedative_Analgesic_Methaqualones,Sedative_Analgesic_Barbiturates,Sedative_Analgesic_Other,Sedative_Analgesic_Dex
0,epinephrine,difetur,hydrocortisone,bemegride,amphetamine,strophantin k,angiotensinamide,chlorothiazide,bumetanide,amiloride,nifedipine,verapamil,benazepril,candesartan,acebutolol,doxazosin,bucindolol,clonidine,aliskiren,fentanyl,amitryptyline,olanzapine,diazepam,diphenhydramine,nitrous oxide,afloqualone,methohexital,suvorexant,dexmedetomidine
1,noradrenaline,izoturon,prednisone,caffeine,atomoxetine,corglycon,amrinone,chlorthalidone,ethacrynic acid,eplerenone,isradipine,diltiazem,captopril,eprosartan,atenolol,phentolamine,carvedilol,guanabenz,bosentan,hydromorphone,trazodone,clozapine,lorazepam,dimenhydrinate,sevoflurane,cloroqualone,thiopental,eszopiclone,NaN
2,phenylephrine,NaN,prednisolone,camphora,bupropion,digoxin,milrinone,hydrochlorothiazide,furosemide,spironolactone,felodipine,NaN,enalapril,irbesartan,betaxolol,indoramin,labetalol,guanfacine,NaN,morphine,mirtazapine,thiothixene,midazolam,doxylamine,halothane,diproqualone,benzylbutylbarbiturate,zaleplon,NaN


In [6]:
cohort.head(2)

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,ICU Site:,ICU Bed#:,MRN:,First Day Enrolled in Study:,First Name:,Last Name:,study_id,mrn,datetime
0,1,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,68,1298742.0,2018-06-06,Charles E,Sullivan,001,1298742,2018-06-06
1,2,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Ellison 4,32,6378271.0,2018-06-11,Derek,Herling,002,6378271,2018-06-11


In [7]:
edw_data = pd.read_csv(edw_data_path) #), dtype=datatype_dict)
# edw_data.fillna(value=-1, inplace=True)
# edw_data = edw_data.astype(datatype_dict)
edw_data['MedicationDSC'] = edw_data['MedicationDSC'].str.lower()
#     edw_data[col] = edw_data[col].str.lower()

edw_data['MedicationTakenDTS'] = pd.to_datetime(edw_data['MedicationTakenDTS'], infer_datetime_format=1)
edw_data = edw_data.drop_duplicates()
# edw_data.rename({'SigTXT': 'SigTXT_Order', 'SigTXT.1': 'SigTXT_Administered'}, axis=1, inplace=True)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (14,16,17,20,23,24,26,27,28,33,34,38) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [8]:
edw_data_1 = edw_data[edw_data.MRN == 3337103]

In [9]:
np.unique([x for x in edw_data_1.MedicationDSC if 'propofol' in x.lower()])

array([], dtype=float64)

In [10]:
edw_data[edw_data.MedicationDSC == 'propofol (diprivan) infusion 10 mg/ml'];

In [11]:
edw_data[edw_data.MedicationDSC == 'propofol 10 mg/ml intravenous emulsion'].head()

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,DoseUnitCD,DoseUnitDSC,MinimumDoseAMT,MaximumDoseAMT,PatientLocationID,PatientLocationDSC,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,SiteCD,SiteDSC,InfusionRateNBR,InfusionRateUnitCD,InfusionRateUnitDSC,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,CommentTxt,DefinedDailyDoseNBR,MorphineEquivalentMGDoseAMT,MorphineEquivalentMGPerHourRateAMT,SigTXT_Order,SigTXT_Order.1,PrescriptionQuantityNBR,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,DiscreteDoseAMT,DiscreteDispenseUnitDSC
50,140298383.0,3631517,3.095235e+09,11150.0,propofol 10 mg/ml intravenous emulsion,2015-10-20 12:07:00.0000000,2015-10-20 11:36:00.0000000,2015-10-20 14:24:00.0000000,NaN,NaN,NaN,NaN,1.003001e+10,BWH PERIOP DEPT,2015-10-20 11:36:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anesthesia,NaN,NaN,NaN,NaN,100.0,NaN,NaN,Anesthesia Stop,As needed,NaN,NaN
51,140298383.0,3631517,3.095235e+09,11150.0,propofol 10 mg/ml intravenous emulsion,2015-10-20 12:07:00.0000000,2015-10-20 11:36:00.0000000,2015-10-20 14:24:00.0000000,NaN,NaN,NaN,NaN,1.003001e+10,BWH PERIOP DEPT,2015-10-20 11:38:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anesthesia,NaN,NaN,NaN,NaN,50.0,NaN,NaN,Anesthesia Stop,As needed,NaN,NaN
52,140298383.0,3631517,3.095235e+09,11150.0,propofol 10 mg/ml intravenous emulsion,2015-10-20 12:07:00.0000000,2015-10-20 11:36:00.0000000,2015-10-20 14:24:00.0000000,NaN,NaN,NaN,NaN,1.003001e+10,BWH PERIOP DEPT,2015-10-20 11:41:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anesthesia,NaN,NaN,NaN,NaN,50.0,NaN,NaN,Anesthesia Stop,As needed,NaN,NaN
432,142521702.0,6188100,3.096298e+09,11150.0,propofol 10 mg/ml intravenous emulsion,2015-10-30 13:23:00.0000000,2015-10-30 11:47:00.0000000,2015-11-02 10:29:00.0000000,NaN,NaN,NaN,NaN,1.003001e+10,BWH PERIOP DEPT,2015-10-30 10:28:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anesthesia,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Anesthesia Stop,As needed,NaN,NaN
433,142521702.0,6188100,3.096298e+09,11150.0,propofol 10 mg/ml intravenous emulsion,2015-10-30 13:23:00.0000000,2015-10-30 11:47:00.0000000,2015-11-02 10:29:00.0000000,NaN,NaN,NaN,NaN,1.003001e+10,BWH PERIOP DEPT,2015-10-30 18:57:00,1.0,Given,Intravenous,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anesthesia,NaN,NaN,NaN,NaN,20.0,NaN,NaN,Anesthesia Stop,As needed,NaN,NaN


In [12]:
edw_data[edw_data.MedicationDSC == 'propofol bolus from infusion 10 mg/ml (premix)'];

In [13]:
adt_data = pd.read_csv(adt_path)
adt_data['EffectiveDTS'] = pd.to_datetime(adt_data['EffectiveDTS'], infer_datetime_format=1)
adt_data = adt_data.sort_values(by=['EffectiveDTS'])

weight_data = pd.read_csv(weight_path)
weight_data = weight_data[(weight_data.FlowsheetMeasureNM=='WEIGHT/SCALE')]
weight_data['weight_kg'] = (weight_data['MeasureTXT'].astype('float')/35.274).round(1)
weight_data['RecordedDTS'] = pd.to_datetime(weight_data['RecordedDTS'], infer_datetime_format=1)
# weight_data['EffectiveDTS'] = pd.to_datetime(adt_data['EffectiveDTS'], infer_datetime_format=1)
# weight_data = adt_data.sort_values(by=['EffectiveDTS'])


In [14]:
weight_data.head(2)

,MRN,PatientID,PatientEncounterID,InpatientDataID,DepartmentID,FlowsheetDataID,FlowsheetMeasureID,FlowsheetMeasureNM,RecordedDTS,MeasureTXT,weight_kg
78,4533154,Z10085576,3.271854e+09,113674078,1.011001e+10,74027805,14,WEIGHT/SCALE,2019-10-01 13:29:00,3120,88.5
118,3243449,Z10251609,3.206128e+09,72638662,1.002001e+10,51945610,14,WEIGHT/SCALE,2018-06-20 15:11:00,1880.08,53.3


In [15]:
# add weight to cohort
cohort['Weight'] = np.nan

for jloc, row in cohort.iterrows():

    mrn_tmp = row.mrn
           
    enrollment_date =  pd.Timestamp(row['datetime'])
    weight_data_mrn = weight_data[weight_data.MRN == mrn_tmp]
    weight_data_mrn_post = weight_data_mrn[weight_data_mrn.RecordedDTS > enrollment_date]
    weight_data_mrn_pre = weight_data_mrn[weight_data_mrn.RecordedDTS < enrollment_date]

    if len(weight_data_mrn_post) > 0:
        cohort.loc[jloc, 'Weight'] = weight_data_mrn_post['weight_kg'].dropna().iloc[0]
    elif len(weight_data_mrn_pre) > 0:
        cohort.loc[jloc, 'Weight'] = weight_data_mrn_pre['weight_kg'].dropna().iloc[-1]
    else:
        print(f'NO WEIGHT FOUND FOR MRN {mrn_tmp}')

In [16]:
cohort['Admission'] = np.nan
cohort['Discharge'] = np.nan

for jloc, row in cohort.iterrows():

    mrn_tmp = row.mrn
           
    enrollment_date =  pd.Timestamp(row['datetime'])
    enrollment_date = enrollment_date.replace(hour=23)

    adt_mrn = adt_data[adt_data.MRN == mrn_tmp]
    cohort['Admission'].loc[jloc] = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Admission') & (adt_mrn.EffectiveDTS<enrollment_date), 'EffectiveDTS'].iloc[-1]
    cohort['Discharge'].loc[jloc] = adt_mrn.loc[(adt_mrn.ADTEventTypeDSC=='Discharge') & (adt_mrn.EffectiveDTS>enrollment_date), 'EffectiveDTS'].iloc[0]

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/core/indexing.py:205: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [17]:
cohort.head(2)

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,ICU Site:,ICU Bed#:,MRN:,First Day Enrolled in Study:,First Name:,Last Name:,study_id,mrn,datetime,Weight,Admission,Discharge
0,1,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,68,1298742.0,2018-06-06,Charles E,Sullivan,001,1298742,2018-06-06,74.8,2018-06-04 21:13:00,2018-06-10 14:01:00
1,2,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Ellison 4,32,6378271.0,2018-06-11,Derek,Herling,002,6378271,2018-06-11,73.6,2018-06-09 02:03:00,2018-06-18 13:49:00


In [18]:
edw_data['OrderStartDTS'] = pd.to_datetime(edw_data['OrderStartDTS'], infer_datetime_format=1)
edw_data['OrderEndDTS'] = pd.to_datetime(edw_data['OrderEndDTS'], infer_datetime_format=1)
edw_data['MedicationTakenDTS'] = pd.to_datetime(edw_data['MedicationTakenDTS'], infer_datetime_format=1)
edw_data = edw_data.sort_values(by='MedicationTakenDTS')

In [19]:
for jloc, row in cohort.iterrows():

    mrn_tmp = row.mrn
    
    index_outside_of_stay_of_interest = edw_data.loc[(edw_data.MRN==mrn_tmp) & ((edw_data.MedicationTakenDTS<row.Admission) | (edw_data.MedicationTakenDTS>row.Discharge))].index
    edw_data.drop(index_outside_of_stay_of_interest, inplace=True)

    index_outside_of_stay_of_interest = edw_data.loc[(edw_data.MRN==mrn_tmp) & ((edw_data.OrderEndDTS<row.Admission) | (edw_data.OrderStartDTS>row.Discharge))].index
    edw_data.drop(index_outside_of_stay_of_interest, inplace=True)    

In [20]:
def get_mapping(df, name1, name2):
    """
    get the dict of df[name1]: df[name2],
    if one-to-many mapping is found, take the first one
    """
    mapping = defaultdict(set)
    for i in range(len(df)):
        mapping[df[name1].iloc[i]].add(df[name2].iloc[i])
        
    for x in mapping:
        if len(mapping[x])>1:
            print(x, mapping[x])
        mapping[x] = list(mapping[x])[0]
        
    return mapping

In [22]:
if save_path_meds_filtered_for_icu_sleep_encounter is not None:
    edw_data.to_csv(save_path_meds_filtered_for_icu_sleep_encounter, index=False)

In [24]:
meds_tostudy = all_meds

In [25]:
df = edw_data.copy()
print(df.shape)

(167402, 39)


In [26]:
found_med_bool = df.MedicationDSC.apply(lambda x: any([x_med in x.lower() for x_med in meds_tostudy]))

In [27]:
# df = df.loc[found_med_bool]
df = df.loc[found_med_bool]

In [28]:
# pd.unique(df.loc[~found_med_bool]['MedicationDSC'])[:100]

In [29]:
df['IsInfusion'] = (df.DiscreteFrequencyDSC=='Continuous PRN') | (df.DiscreteFrequencyDSC=='Continuous') #  | ~pd.isna(df.DurationNBR)
for order_id in pd.unique(df.OrderID):
    
    df_order = df[df.OrderID == order_id]
    if (np.all(pd.isna(df_order.DiscreteDoseAMT)) & (~np.any(pd.isna(df_order.InfusionRateNBR)))):
        index_order = df_order.index
        df.loc[index_order, 'IsInfusion'] = True
        
df['unit'] = np.nan
df['dose'] = np.nan


# manual correction:
# df.loc[df.MedicationDSC == 'phenylephrine (NEO-SYNEPHRINE) 80 mcg/mL infusion bag', 'DoseUnitDSC'] = 'mcg/min'


Number of non-valid entries: 68
Remove due to missing infusion entry: 920
Removing the following ear/eye drop meds: 
 ['gatifloxacin 0.5 %-prednisolone 1 %-bromfenac 0.075 % eye drops,suspen'
 'prednisolone acetate 1 % eye drops,suspension'
 'timolol maleate 0.25 % eye drops'
 'ciprofloxacin 0.3 %-dexamethasone 0.1 % ear drops,suspension'
 'timolol maleate 0.5 % eye drops'].
Removing the following spray meds: 
 ['phenylephrine 0.25 % nasal spray'].
Removing the following millicurie-unit meds: 
 ['xenon xe 133 gas 10 mci'].


In [30]:
np.unique([x for x in df.DoseUnitDSC.dropna() if 'kg' in x])

array(['mL/kg', 'mcg/kg', 'mcg/kg/hr', 'mcg/kg/min'], dtype='<U10')

In [31]:
df.loc[np.isin(df.DoseUnitDSC, ['mcg/kg/min', 'mcg/kg/hr'])].MedicationDSC.unique()

array(['propofol 10 mg/ml intravenous emulsion',
       'dexmedetomidine infusion 4 mcg/ml (50 ml)',
       'ketamine infusion syr in swfi 10 mg/ml (50 ml) cmpd mgh',
       'dexmedetomidine 200 mcg/50 ml (4 mcg/ml) in 0.9 % sodium chloride iv',
       'dobutamine 500 mg/250 ml (2,000 mcg/ml) in 5 % dextrose iv',
       'dexmedetomidine infusion 4 mcg/ml (250 ml)',
       'ketamine infusion 10 mg/ml in ns (50 ml) cmpd mgh',
       'dexmedetomidine 400 mcg/100 ml (4 mcg/ml) in 0.9 % sodium chloride iv',
       'dexmedetomidine infusion 4 mcg/ml (150 ml) cmpd mgh',
       'dopamine 200 mg/250 ml (800 mcg/ml) in 5 % dextrose intravenous soln',
       'ketamine infusion in ns 10 mg/ml (100 ml) cmpd mgh'], dtype=object)

In [32]:
df.loc[np.isin(df.DoseUnitDSC, ['mL/kg', 'mcg/kg'])].MedicationDSC.unique()

array(['id-dexmedetomidine or placebo (2017p000090) bolus 80 mcg/20 ml',
       'id-dexmedetomidine or placebo (2017p000090) infusion 80 mg/20 ml',
       'id-dexmedetomidine 1.33 mcg/ml or 4 mcg/ml or placebo (2017p000090) infusion'],
      dtype=object)

In [33]:
df.loc[np.isin(df.DoseUnitDSC, ['mL/hr', 'mL/min'])].MedicationDSC.unique()

array(['hydromorphone (pf) 20 mcg/ml-bupivacaine 0.1 % epid bag in ns (100 ml) cmpd mgh',
       'fentanyl-bupivacaine (pf) 2 mcg/ml-0.1 % epid bag in ns (100 ml) cmpd mgh'],
      dtype=object)

In [34]:
# define equivalent doses (scaler is inverse of those values): 
morphine_ratios = {
    'morphine_parenteral': 1, 
    'morphine_oral': 3,
    'codeine': 20,
    'hydromorphone_parenteral': 0.15,
    'hydromorphone_oral': 0.75,
    'oxycodone_oral': 2,
    'hydrocodone_oral': 3,
    'oxymorphone_parenteral': 0.1,
    'oxymorphone_oral': 0.15,
    'levorphanol_parenteral': 0.2,
    'levorphanol_oral': 0.4,
    'methadone_parenteral': 1,
    'methadone_oral': 2,
    'fentanyl_parenteral': 0.01,
    'fentanyl_transdermal': 7.2,
    'fentanyl_oral': 0.01,
    'buphrenorphine_parenteral': 0.03,
    'buphrenorphine_transdermal': 0.001,
    'tapentadol': 0.75,
    'tramadol': 10,
    'meperidine': 30,
    'pentazocine': 18,
    'propoxyphene': 6.5,
    'nalbuphine': 0.25,
    'remifentanil_parenteral': 0.01,
    'sufentanil_parenteral': 0.001,
}

# define equivalent doses (scaler is inverse of those values): 
benzos_ratios = {
    'alprazolam': 0.5, 
    'bromazepam': 5, 
    'chlordiazepoxide': 25, 
    'clonazepam': 0.5, 
    'clorazepate': 15, 
    'diazepam': 10, 
    'estazolam': 1.5, 
    'flunitrazepam': 1, 
    'flurazepam': 22, 
    'halazepam': 20, 
    'ketazolam': 7.5, 
    'lorazepam': 1, 
    'medazepam': 10, 
    'midazolam': 2, 
    'nitrazepam': 5, 
    'nordazepam': 10, 
    'oxazepam': 30, 
    'prazepam': 15, 
    'quazepam': 15, 
    'temazepam': 20, 
    'triazolam': 0.25, 
    'eszopiclone': 15, 
    'zaleplon': 20, 
    'zolpidem': 20, 
    'butalbital': 50, 
    'pentobarbital': 100, 
    'phenobarbital': 30, 
    'meprobamate': 400, 
    'carisoprodol': 350, 
    'choral hydrate': 250, 
    'gamma_hydroxybutyrate': 1, 
}


antipsychotics_ratios = {
    'chlorpromazine': 100, 
    'fluphenazine': 2, 
    'haloperidol': 2, 
    'loxapine': 10, 
    'perphenazine': 8, 
    'pimozide': 2, 
    'prochlorperazine': 15, 
    'trifluoperazine': 3.5, 
    'thioridazine': 100, 
    'thiothixene': 4, 
    'aripiprazole': 7.5, 
    'asenapine': 4, 
    'brexpiprazole': 1, 
    'cariprazine': 1,
    'clozapine': 100, 
    'iloperidone': 3.5, 
    'lurasidone': 16, 
    'olanzapine': 5, 
    'paliperidone': 2, 
    'quetiapine': 75, 
    'risperidone': 1, 
    'ziprasidone': 60, 
}



benzo_meds = ['diazepam','lorazepam','midazolam','clonazepam','diazepam','estazolam',
 'flunitrazepam','lorazepam','midazolam','nitrazepam','oxazepam',
 'triazolam','temazepam','chlordiazepoxide','alprazolam','clobazam',
 'clorazepate','etizolam']
antipsychotic_meds = ['olanzapine','clozapine','thiothixene','haloperidol','fluphenazine',
 'prochlorperazine','trifluoperazine','loxapine','quetiapine','asenapine']
opioid_meds = ['fentanyl','hydromorphone','morphine','remifentanil','tramadol',
 'tapentadol','oxymorphone','oxycodone','hydrocodone','methadone',
 'propoxyphene','meperidine','codeine','alfentanil','sufentanil']


# OPIOID DICTIONARIES:
opioid_meds_category = {}
opioid_meds_family = {}
opioid_meds_ratios = {}

meds_all = pd.unique(df.MedicationDSC)
for med in opioid_meds:
#     print(f'\n{med}:')
    opioid_dsc = [x for x in meds_all if med in x]
    for med_tmp in opioid_dsc:
        if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
            cat = 'parenteral'
        elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
            cat = 'transdermal'
        elif any(['oral', 'tablet']):
            cat = 'oral'
        else:
            print(f'No category found for {med}!')
            
        opioid_meds_category[med_tmp] = cat
        opioid_meds_family[med_tmp] = med
        
        if med + '_' + cat in morphine_ratios:
            opioid_meds_ratios[med_tmp] = morphine_ratios[med + '_' + cat]
        elif med in morphine_ratios:
            opioid_meds_ratios[med_tmp] = morphine_ratios[med]
        else:
            print(f'{med} is not found in morphine_ratios!')
            
            
# BENZOS DICTIONARIES:
benzos_meds_category = {}
benzos_meds_family = {}
benzos_meds_ratios = {}

for med in benzo_meds:
#     print(f'\n{med}:')
    benzos_dsc = [x for x in meds_all if med in x]
    for med_tmp in benzos_dsc:
        if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
            cat = 'parenteral'
        elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
            cat = 'transdermal'
        elif any(['oral', 'tablet']):
            cat = 'oral'
        else:
            print(f'No category found for {med}!')
            
        benzos_meds_category[med_tmp] = cat
        benzos_meds_family[med_tmp] = med
        
#         if med + '_' + cat in benzos_ratios: # TODO CAT=ROUTE, like in opioid.
#             benzos_meds_ratios[med_tmp] = benzos_ratios[med + '_' + cat]
        if med in benzos_ratios:
            benzos_meds_ratios[med_tmp] = benzos_ratios[med]
        else:
            print(f'{med} is not found in morphine_ratios!')

            
            
# ANTIPSYCH DICTIONARIES:
antipsychotics_meds_category = {}
antipsychotics_meds_family = {}
antipsychotics_meds_ratios = {}

for med in antipsychotic_meds:
#     print(f'\n{med}:')
    antipsychotics_dsc = [x for x in meds_all if med in x]
    for med_tmp in antipsychotics_dsc:
        if any(['injection' in med_tmp, 'intravenous' in med_tmp, 'infusion' in med_tmp, 
                'pcea'in med_tmp, 'pca' in med_tmp, 'bag' in med_tmp, 'bolus' in med_tmp, 'ivpb' in med_tmp]):
            cat = 'parenteral'
        elif any(['transdermal' in med_tmp, 'patch' in med_tmp]):
            cat = 'transdermal'
        elif any(['oral', 'tablet']):
            cat = 'oral'
        else:
            print(f'No category found for {med}!')
            
        antipsychotics_meds_category[med_tmp] = cat
        antipsychotics_meds_family[med_tmp] = med
        
#         if med + '_' + cat in benzos_ratios: # TODO CAT=ROUTE, like in opioid.
#             benzos_meds_ratios[med_tmp] = benzos_ratios[med + '_' + cat]
        if med in antipsychotics_ratios:
            antipsychotics_meds_ratios[med_tmp] = antipsychotics_ratios[med]
        else:
            print(f'{med} is not found in antipsychotics_ratios!')
        

In [36]:
def change_dtype_and_colname(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        df.rename({col: col.replace('/', 'per').replace('-','_').replace(' ', '_').replace('%', 'perc')}, axis=1, inplace=True)

    return df

def compute_opioid_sum(df):
    
    df['opioid_sum'] = 0
    for med_tmp in opioid_meds_ratios:
        med_tmp_colname = med_tmp.replace('/', 'per').replace('-','_').replace(' ', '_').replace('%', 'perc')
        if not med_tmp_colname in df.columns: 
            continue
        df['opioid_sum'] += df[med_tmp_colname] /  opioid_meds_ratios[med_tmp]
        
    return df

def compute_benzos_sum(df):

    df['benzos_sum'] = 0
    for med_tmp in benzos_meds_ratios:
        med_tmp_colname = med_tmp.replace('/', 'per').replace('-','_').replace(' ', '_').replace('%', 'perc')
        if not med_tmp_colname in df.columns: 
            continue
        df['benzos_sum'] += df[med_tmp_colname] /  benzos_meds_ratios[med_tmp]

    return df

def compute_antipsychotics_sum(df):
        
    df['antipsychotics_sum'] = 0
    antipsychotic_dsc = []

    for med in antipsychotic_meds:
    #     print(f'\n{med}:')
        antipsychotic_dsc += [x for x in meds_all if med in x]

    for med_tmp in antipsychotic_dsc:
        med_tmp_colname = med_tmp.replace('/', 'per').replace('-','_').replace(' ', '_').replace('%', 'perc')
        if not med_tmp_colname in meds_ts_df_mrn.columns: 
            continue
    #     print(f'{med_tmp_colname}')
        df['antipsychotics_sum'] += df[med_tmp_colname] /  antipsychotics_meds_ratios[med_tmp]

    return df

    
def create_hdr():
    
    hdr = {}
    hdr['study_id'] = int(row_subject.study_id)
    hdr['MRN'] = int(row_subject.mrn)
    hdr['fs'] = fs_med
    hdr['start_datetime'] = np.array([meds_ts_df_mrn.index[0].year,
                                     meds_ts_df_mrn.index[0].month,
                                     meds_ts_df_mrn.index[0].day,
                                     meds_ts_df_mrn.index[0].hour,
                                     meds_ts_df_mrn.index[0].minute,
                                     meds_ts_df_mrn.index[0].second,
                                     meds_ts_df_mrn.index[0].microsecond])
    
    return hdr
    

In [39]:
df

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,DoseUnitCD,DoseUnitDSC,MinimumDoseAMT,MaximumDoseAMT,PatientLocationID,PatientLocationDSC,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,SiteCD,SiteDSC,InfusionRateNBR,InfusionRateUnitCD,InfusionRateUnitDSC,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,CommentTxt,DefinedDailyDoseNBR,MorphineEquivalentMGDoseAMT,MorphineEquivalentMGPerHourRateAMT,SigTXT_Order,SigTXT_Administered,PrescriptionQuantityNBR,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,DiscreteDoseAMT,DiscreteDispenseUnitDSC,IsInfusion,unit,dose
84941,416610917.0,1298742,3.203969e+09,701192.0,phenylephrine (neo-synephrine) 80 mcg/ml infus...,2018-06-05 07:30:00.0000000,2018-06-05 07:30:00,2018-06-05 09:30:00,NaN,NaN,NaN,NaN,NaN,NaN,2018-06-05 07:20:00,14.0,Rate Verify,NaN,NaN,NaN,NaN,45.0,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
85043,416610917.0,1298742,3.203969e+09,701192.0,phenylephrine (neo-synephrine) 80 mcg/ml infus...,2018-06-05 07:30:00.0000000,2018-06-05 07:30:00,2018-06-05 09:30:00,NaN,NaN,NaN,NaN,NaN,NaN,2018-06-05 08:00:00,9.0,Rate Change,NaN,NaN,NaN,NaN,112.5,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,150.0,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
85044,416610917.0,1298742,3.203969e+09,701192.0,phenylephrine (neo-synephrine) 80 mcg/ml infus...,2018-06-05 07:30:00.0000000,2018-06-05 07:30:00,2018-06-05 09:30:00,NaN,NaN,NaN,NaN,NaN,NaN,2018-06-05 08:15:00,9.0,Rate Change,NaN,NaN,NaN,NaN,225.0,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
84196,416643169.0,1298742,3.203969e+09,701192.0,phenylephrine (neo-synephrine) 80 mcg/ml infus...,2018-06-05 08:22:00.0000000,2018-06-05 08:22:00,2018-06-05 08:30:00,NaN,NaN,NaN,NaN,NaN,NaN,2018-06-05 08:30:00,116.0,Override Pull,Intravenous,11.0,NaN,NaN,0.0,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
84158,416643171.0,1298742,3.203969e+09,701192.0,phenylephrine infusion 80 mcg/ml (250 ml) cmpd...,2018-06-05 08:23:00.0000000,2018-06-05 09:15:00,2018-06-07 10:21:00,28.0,mcg/min,0.0,1000.0,1.002001e+10,MGH BLAKE 7 MICU,2018-06-05 08:30:00,116.0,Override Pull,Intravenous,11.0,NaN,NaN,0.0,41.0,mL/hr,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Continuous,0-1000,NaN,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443253,628505262.0,1849492,3.275833e+09,8085.0,trazodone 50 mg tablet,2019-12-26 13:27:00.0000000,2019-12-26 21:00:00,2020-01-06 15:00:00,3.0,mg,50.0,NaN,1.002001e+10,MGH BIGELOW 13 RACU,2020-01-05 21:00:00,1.0,Given,Gastrostomy Tube,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,50.0,NaN,NaN,Patient Discharge,Nightly,50,NaN,False,NaN,NaN
443298,628505283.0,1849492,3.275833e+09,8528.0,verapamil 120 mg tablet,2019-12-29 10:40:00.0000000,2019-12-29 13:00:00,2020-01-06 15:00:00,3.0,mg,90.0,NaN,1.002001e+10,MGH ELLISON19 THOR/VAS,2020-01-06 00:48:00,1.0,Given,Gastrostomy Tube,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,90.0,NaN,NaN,Patient Discharge,4 times daily,90,NaN,False,NaN,NaN
443299,628505283.0,1849492,3.275833e+09,8528.0,verapamil 120 mg tablet,2019-12-29 10:40:00.0000000,2019-12-29 13:00:00,2020-01-06 15:00:00,3.0,mg,90.0,NaN,1.002001e+10,MGH ELLISON19 THOR/VAS,2020-01-06 05:36:00,1.0,Given,Gastrostomy Tube,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,90.0,NaN,NaN,Patient Discharge,4 times daily,90,NaN,False,NaN,NaN
442800,627285323.0,1849492,3.275833e+09,9071.0,amlodipine 5 mg tablet,2019-12-23 11:27:00.0000000,2019-12-24 09:00:00,2020-01-06 15:00:00,3.0,mg,5.0,NaN,1.002001e+10,MGH BIGELOW 13 RACU,2020-01-06 08:17:00,1.0,Given,Gastrostomy Tube,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,5.0,NaN,NaN,Patient Discharge,Daily,5,NaN,False,NaN,NaN


In [32]:
# for jloc, row in df.iterrows():
#     if ~pd.isna(row.DoseUnitDSC):
#         df.loc[jloc, 'unit'] = row.DoseUnitDSC
    
df.loc[~pd.isna(df.DoseUnitDSC), 'unit'] = df.loc[~pd.isna(df.DoseUnitDSC), 'DoseUnitDSC']



# for some of them, infusion rate is missing. mainly due to patient adminsitered dose. rest is removed here.
df_missing_infusion = df[(df.IsInfusion==1) & (pd.isna(df.InfusionRateNBR))]
meds_self_controlled = [x for x in pd.unique(df_missing_infusion.MedicationDSC) if ('PCA'.lower() in x) | ('PCEA'.lower() in x)]
print(f'For the following meds, it patient-self-administration is assumed because of "PCA" or "PCEA" in Med Name. Placeholder dose of 0.01 is imputed.')
print(meds_self_controlled)
for med_self_controlled in meds_self_controlled:

    df.loc[df.MedicationDSC == med_self_controlled, 'unit'] = 'mg'    
    df.loc[(df.MedicationDSC == med_self_controlled) & (df.MARActionDSC == 'Stopped'), 'dose'] = 0
    df.loc[(df.MedicationDSC == med_self_controlled) & (df.MARActionDSC == 'Stopped'), 'unit'] = 'mg'     
    
    

For the following meds, it patient-self-administration is assumed because of "PCA" or "PCEA" in Med Name. Placeholder dose of 0.01 is imputed.
[]


In [78]:
med_unit_dict = {}

for med in df.MedicationID.unique():
    
    # check if for this med, only one unique unit is used for all patients:
    unique_unit = None
    units_for_this_med = pd.unique(df[df.MedicationID == med].DoseUnitDSC.dropna())
    med_unit_dict[med] = units_for_this_med
    if len(units_for_this_med) == 1:
        df.loc[df.MedicationID == med, 'unit'] = units_for_this_med
    
    elif len(units_for_this_med) == 0:
        print(f'0: {med}')

    

>1: 909247.0
0: 707703.0
0: 700995.0
0: 709543.0
0: 701806.0
>1: 2444.0
0: 701599.0
0: 10734.0
>1: 909916.0


In [83]:
df[pd.isna(df.unit)]

,OrderID,MRN,PatientEncounterID,MedicationID,MedicationDSC,OrderInstantDTS,OrderStartDTS,OrderEndDTS,DoseUnitCD,DoseUnitDSC,MinimumDoseAMT,MaximumDoseAMT,PatientLocationID,PatientLocationDSC,MedicationTakenDTS,MARActionCD,MARActionDSC,RouteDSC,RouteCD,SiteCD,SiteDSC,InfusionRateNBR,InfusionRateUnitCD,InfusionRateUnitDSC,DurationNBR,DurationUnitCD,DurationUnitDSC,ActionSourceDSC,CommentTxt,DefinedDailyDoseNBR,MorphineEquivalentMGDoseAMT,MorphineEquivalentMGPerHourRateAMT,SigTXT_Order,SigTXT_Administered,PrescriptionQuantityNBR,MedicationDiscontinueReasonDSC,DiscreteFrequencyDSC,DiscreteDoseAMT,DiscreteDispenseUnitDSC,IsInfusion,unit,dose
88943,419008292.0,6378271,3.204669e+09,707703.0,morphine pca bag in ns 1 mg/ml (50 ml) dp_cmpd...,2018-06-11 18:00:00.0000000,2018-06-11 19:00:00,2018-06-11 18:28:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH ELLISON 4 SICU,2018-06-11 18:17:00,6.0,New Bag,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
88256,419008300.0,6378271,3.204669e+09,707703.0,morphine pca bag in ns 1 mg/ml (50 ml) dp_cmpd...,2018-06-11 22:32:00.0000000,2018-06-11 23:30:00,2018-06-12 13:39:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH ELLISON 4 SICU,2018-06-11 18:59:00,6.0,New Bag,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
88257,419008300.0,6378271,3.204669e+09,707703.0,morphine pca bag in ns 1 mg/ml (50 ml) dp_cmpd...,2018-06-11 22:32:00.0000000,2018-06-11 23:30:00,2018-06-12 13:39:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH ELLISON 4 SICU,2018-06-12 03:11:00,6.0,New Bag,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
88329,419008300.0,6378271,3.204669e+09,707703.0,morphine pca bag in ns 1 mg/ml (50 ml) dp_cmpd...,2018-06-11 22:32:00.0000000,2018-06-11 23:30:00,2018-06-12 13:39:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH ELLISON 4 SICU,2018-06-12 12:04:00,6.0,New Bag,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
88307,419008333.0,6378271,3.204669e+09,707703.0,morphine pca bag in ns 1 mg/ml (50 ml) dp_cmpd...,2018-06-12 13:39:00.0000000,2018-06-12 14:30:00,2018-06-13 06:52:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH ELLISON 4 SICU,2018-06-12 13:44:00,9.0,Rate Change,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428880,616038491.0,6175987,3.276934e+09,709543.0,hydromorphone (pf) pca bag in ns 1 mg/ml (25 m...,2019-11-26 01:36:00.0000000,2019-11-26 02:30:00,2019-12-03 11:17:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH BLAKE 12 ICU,2019-12-02 01:25:00,6.0,New Bag,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
428881,616038491.0,6175987,3.276934e+09,709543.0,hydromorphone (pf) pca bag in ns 1 mg/ml (25 m...,2019-11-26 01:36:00.0000000,2019-11-26 02:30:00,2019-12-03 11:17:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH BLAKE 12 ICU,2019-12-02 08:41:00,14.0,Rate Verify,Intravenous (PCA General),220.0,NaN,NaN,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
428882,616038491.0,6175987,3.276934e+09,709543.0,hydromorphone (pf) pca bag in ns 1 mg/ml (25 m...,2019-11-26 01:36:00.0000000,2019-11-26 02:30:00,2019-12-03 11:17:00,NaN,NaN,NaN,NaN,1.002001e+10,MGH BLAKE 12 ICU,2019-12-02 16:32:00,6.0,New Bag,Intravenous (PCA General),220.0,8.0,Port,0.01,NaN,NaN,NaN,NaN,NaN,MAR,NaN,NaN,NaN,NaN,0.01,NaN,NaN,NaN,Continuous,NaN,NaN,True,NaN,NaN
428883,616038491.0,6175987,3.276934e+09,709543.0,hydromorphone (pf) pca bag in ns 1 mg/ml (25 m...,2019-11-26 01:36:00.0000000,2019-11-2

In [65]:
df[pd.isna(df.unit)];

In [68]:
print(df[df.MedicationID == 2444.0].MedicationDSC.unique())
print(df[df.MedicationID == 2444.0].DoseUnitDSC.unique())

['digoxin 125 mcg tablet' 'digoxin 250 mcg tablet'
 'digoxin 125 mcg (0.125 mg) tablet']
['mg' 'mcg']


In [61]:
df[df.MedicationID == 10734.0].MedicationDSC.iloc[0]

'norepinephrine (levophed) 1 mg/ml injection'

In [62]:
df[df.MedicationID == 909916.0].MedicationDSC.iloc[0]

'id-dexmedetomidine or placebo (2017p000090) infusion 4 mcg/ml or 1.33 mcg/ml'

In [ ]:

    
    if len(units_for_this_med) == 1:
        unique_unit = units_for_this_med[0]
    elif len(units_for_this_med) == 0:
        # first try to find unit from other med-name-spelling but same MedID:
        med_id = df_mrn_med.MedicationID.iloc[0]
        df_med_id = df[df.MedicationID == med_id]
        units_for_this_med = df_med_id.DoseUnitDSC.dropna().unique()
        if len(units_for_this_med) == 1:
            unique_unit = units_for_this_med[0]
    if unique_unit is None:
        # if not working, extract the unit from the medname:
        unique_unit = extract_unit_from_med_name(df[df.MedicationDSC == med])
        if unique_unit is None:
            print('unit extraction from med name failed.')
            print(med)


In [106]:
# some infusions are in mL/hr unit, change that to mg/hr or mcg/hr here:
no_dose_found = []
for jloc, row in df[df.DoseUnitDSC == 'mL/hr'].iterrows():
    unit_dose = re.search(r"\d+ .+/ml", row.MedicationDSC)
    if unit_dose is None:
        if not row.MedicationDSC in no_dose_found:
            no_dose_found.append(row.MedicationDSC)
            print(f'No dose found, therefore skip: {row.MedicationDSC}')
        continue
    unit_dose = unit_dose[0]
    assert ~pd.isna(row.DiscreteDoseAMT)
    dose = np.float(unit_dose.split(' ')[0]) * np.float(row.DiscreteDoseAMT)
    unit = unit_dose.split(' ')[1]
    assert '/ml' in unit
    unit = unit.replace('/ml', '/hr')
    
    df.loc[jloc, 'dose'] = dose
    df.loc[jloc, 'unit'] = unit

In [96]:
# infusion that are weight dependent (UnitDose with /kg) follow a general pattern too, we can transform the Units to mg/h here too.
pd.isna(df.loc[np.isin(df.DoseUnitDSC, ['mcg/kg/min', 'mcg/kg/hr'])].SigTXT_Order).sum()

for jloc, row in df[np.isin(df.DoseUnitDSC, ['mcg/kg/min', 'mcg/kg/hr'])].iterrows():
    
    unit_dose = re.search(r"\d+ .{1,3}/ml?", row.MedicationDSC)
    if unit_dose is None:
        if not row.MedicationDSC in no_dose_found:
            no_dose_found.append(row.MedicationDSC)
            print(f'No dose found, therefore skip: {row.MedicationDSC}')
        continue
    unit_dose = unit_dose[0]
    # make sure units match (e.g. /ml from MedicationDSC and ml/ from Infusionrate)
    unit.split('/')[1].lower() == row.InfusionRateUnitDSC.split('/')[0].lower()

    dose = np.float(unit_dose.split(' ')[0]) * np.float(row.InfusionRateNBR)
    unit = unit_dose.split(' ')[1]
    assert '/ml' in unit
    unit = unit.replace('/kg', '')
    
    df.loc[jloc, 'dose'] = dose
    df.loc[jloc, 'unit'] = unit


In [38]:
# antipsychotic_dsc = []
# for med in antipsychotic_meds:
# #     print(f'\n{med}:')
#     antipsychotic_dsc += [x for x in meds_all if med in x]

In [60]:
overwrite = True

meds_ts_df_mrn_all = []

for jloc_subject, row_subject in cohort.iterrows():

    mrn = row_subject.mrn
    weight_kg = row_subject.Weight

    df_mrn = df[df.MRN == mrn]
    meds_mrn = pd.unique(df_mrn['MedicationDSC']);
    min_date = np.nanmin([df_mrn.OrderStartDTS.values, df_mrn.MedicationTakenDTS.values])
    max_date = np.nanmax([df_mrn.OrderEndDTS.values, df_mrn.MedicationTakenDTS.values])

    # initialize new timeseries-based df for this MRN:
    fs_med = 1
    meds_ts_df_mrn = pd.DataFrame(index=pd.date_range(min_date, max_date, freq= str(fs_med)+'s')) 

    for med in meds_mrn:
#         print(med)

        if not 'PLACEBO'.lower() in med:
            meds_ts_df_mrn[med] = 0
            
        if '<remdesivir>' in med:
            continue
            
        df_mrn_med = df_mrn[df_mrn.MedicationDSC == med]



        # FIRST TYPE OF MEDS: NON CONT. INFUSIONS:
        for jloc, row in df_mrn_med.iterrows():
            
            if 'PLACEBO'.lower() in med: continue
            
            isinfusion = row.IsInfusion

            if not isinfusion:

                dose = row.DiscreteDoseAMT
                if (pd.isna(dose)) | (type(dose) != float):
                    dose = row.SigTXT_Order
                dose = np.float(dose)

                if unique_unit is not None:
                    unit = unique_unit
                else:
                    unit = row.DoseUnitDSC
                
                if unit == 'patch':
                    continue

                if (pd.isna(dose)) | (pd.isna(unit)):
                    if (pd.isna(row.MedicationTakenDTS)) | (row.MARActionDSC != 'Given'):
                        # medication not given anyway, skip.
                        continue
                    print(f'For given Med, Dose or Unit is NAN:  Dose: {dose},  Unit: {unit}  , OrderID {row.OrderID}, medication: {row.MedicationDSC}')
                    continue
                    
                dose = convert_units(dose, unit)
                meds_ts_df_mrn.loc[row.MedicationTakenDTS, med] = dose
                
        # SECOND TYPE OF MEDS: CONT. INFUSIONS:
        df_mrn_med_infusion = df_mrn_med.loc[df_mrn_med.IsInfusion==True]
        order_ids = pd.unique(df_mrn_med_infusion.OrderID)
        
        # fill in infusion rate for each order:
        for order_id in order_ids:
            
            if 'PLACEBO'.lower() in med: continue

            df_order = df_mrn_med_infusion[df_mrn_med_infusion.OrderID == order_id]
            
            if len(df_order[df_order['InfusionRateNBR']>0]) == 0:
                continue
            start_time_order = df_order[df_order['InfusionRateNBR']>0].iloc[0].MedicationTakenDTS
            end_time_order = df_order.iloc[-1].OrderEndDTS
            if np.sum( meds_ts_df_mrn.loc[start_time_order : end_time_order, med].dropna() >0 ):
                print(f'Overlapping Order Start and End for: MRN={mrn}, MedicationDSC={row.MedicationDSC}, start_time_order={start_time_order}, end_time_order={end_time_order}')
            meds_ts_df_mrn.loc[start_time_order : end_time_order, med] = np.nan

            if df_order.iloc[-1].InfusionRateNBR == 0:
                end_time_order = df_order.iloc[-1].MedicationTakenDTS
            df_order = df_order[(df_order.MedicationTakenDTS >= start_time_order) & (df_order.MedicationTakenDTS <= end_time_order)]

            for jloc, row in df_order.iterrows():
                
                dose = row.SigTXT_Order
                if unique_unit is not None:
                    unit = unique_unit
                else:
                    unit = row.DoseUnitDSC

                dose = convert_units(dose, unit)
                meds_ts_df_mrn.loc[row.MedicationTakenDTS, med] = dose
        
        # THIRD TYPE OF MEDS: PATCHES:
        df_mrn_med_patch = df_mrn_med.loc[df_mrn_med.DoseUnitDSC=='patch']
        order_ids = pd.unique(df_mrn_med_patch.OrderID)
        
        for order_id in order_ids:

            if 'PLACEBO'.lower() in med: continue

            df_order = df_mrn_med_patch[df_mrn_med_patch.OrderID == order_id]
            if not 'Patch Applied' in df_order.MARActionDSC.values:
                print('no patch applied?')
                continue

            start_time_order = df_order[df_order['MARActionDSC'] == 'Patch Applied'].iloc[0].MedicationTakenDTS
            end_time_order = df_order.iloc[-1].OrderEndDTS
            
            if df_order.iloc[-1].MARActionDSC == 'Patch Removed':
                end_time_order = df_order.iloc[-1].MedicationTakenDTS
            
            meds_ts_df_mrn[med].loc[start_time_order : end_time_order] = np.nan
            
            for jloc, row in df_order.iterrows():

                if row.MARActionDSC in ['Patch Removed', 'Due']:
                    dose = 0
                    unit = 'mcg/hr'

                elif row.MARActionDSC in ['Patch Applied']:
                    unit_dose_patch = re.search(r"(\d+[.])?\d+ .+? ", row.MedicationDSC)[0]
                    dose = np.float(unit_dose_patch.split(' ')[0])
                    unit = unit_dose_patch.split(' ')[1]
                    if '/24' in unit: # that means descriptions says m(c)g/24 hours.
                        dose = dose/24
                        unit = unit.replace('24', 'hr')

                else:
                    print(f'Unknown Patch MARActionDSC: {row.MARActionDSC}. Skip.')
                    continue
                
                dose = convert_units(dose, unit)
                meds_ts_df_mrn.loc[row.MedicationTakenDTS, med] = dose

            # if last entry has dose>0, there is no end indicated. use maximum duration derived from DiscreteFrequency. 
            # I think this is necessary because some Orders for patches are for a month, without any entry after a few days any more.
            if dose > 0:
                try:
                    duration = np.float(re.search('\d+', row.DiscreteFrequencyDSC)[0])
                    unit_duration = [x for x in ['minute', 'hour', 'day'] if x in row.DiscreteFrequencyDSC.lower()][0]
                except:
                    print('Patch max duration parsing failed.')

                if unit_duration == 'minute':
                    duration = duration/60
                elif unit_duration == 'hour':
                    duration = duration
                elif unit_duration == 'day':
                    duration = duration*24
                else:
                    print(f'Unexpected duration unit for Patch {unit_duration}')

                end_dt = row.MedicationTakenDTS + timedelta(hours=duration)
                meds_ts_df_mrn.loc[end_dt, med] = 0


        # special type of meds. e.g. study drug w/ Placebo for ICU-Sleep:
        # do not convert unit for study drug as we are blinded. keep ML/h. can be easily multiplied by mcg/ml once we are unblinded.
        if '2017P000090'.lower() in med:
            order_ids = pd.unique(df_mrn_med.OrderID)
            study_drug_name = 'DEXMEDETOMIDINE OR PLACEBO (2017P000090)'.lower()
            if not study_drug_name in meds_ts_df_mrn.columns:
                meds_ts_df_mrn[study_drug_name] = np.nan
                meds_ts_df_mrn.loc[min_date, study_drug_name] = 0

    
            for order_id in order_ids:
        
                df_order = df[df.OrderID == order_id]

                for jloc, row in df_order.iterrows():

                    dose = row.InfusionRateNBR
                    unit = row.InfusionRateUnitDSC
                    if dose > 0:

                        if row.DurationUnitDSC == 'Minutes':
                            duration_minutes = np.float(row.DurationNBR)
                        elif row.DurationUnitDSC == 'Hours':
                            duration_minutes = np.float(row.DurationNBR) * 60
                        else:
                            print(f'Unexpected DurationUnitDSC for ICU-Sleep study drug: {row.DurationUnitDSC}')

                        end_dt = row.MedicationTakenDTS + timedelta(minutes=duration_minutes)

                        meds_ts_df_mrn.loc[row.MedicationTakenDTS, study_drug_name] = dose
                        meds_ts_df_mrn.loc[end_dt, study_drug_name] = 0 # set definitive end point.

                    else:
                        meds_ts_df_mrn.loc[row.MedicationTakenDTS, study_drug_name] = dose
        
            med = study_drug_name
            
        # this med is done, completing time series with forward-propagate/fill NA values.
        meds_ts_df_mrn[med].fillna(method='ffill', inplace=True)
    
    meds_ts_df_mrn = change_dtype_and_colname(meds_ts_df_mrn)

    meds_ts_df_mrn = compute_opioid_sum(meds_ts_df_mrn)
    meds_ts_df_mrn = compute_benzos_sum(meds_ts_df_mrn)
    meds_ts_df_mrn = compute_antipsychotics_sum(meds_ts_df_mrn)

    if project == 'icu-sleep':
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, 'icu_' + str(row_subject.study_id) + '_meds.h5')
    elif project == 'sample':
        row_subject['study_id'] = 0
        hdr = create_hdr() 
        file_output_path = os.path.join(meds_all_output_dir, str(row_subject.mrn) + '_meds.h5')

    write_to_hdf5_file(meds_ts_df_mrn, file_output_path, hdr=hdr, default_dtype='float32', overwrite=overwrite)
    meds_ts_df_mrn_all.append(meds_ts_df_mrn)

Overlapping Order Start and End for: MRN=3337103, MedicationDSC=propofol 10 mg/ml intravenous emulsion, start_time_order=2020-05-07 07:48:00, end_time_order=2020-05-08 17:57:00
unit extraction from med name failed.
vancomycin ivpb in 250 ml
unit extraction from med name failed.
vancomycin ivpb in 250 ml
unit extraction from med name failed.
sodium bicarbonate 8.4 % injection syringe
For given Med, Dose or Unit is NAN:  Dose: 50.0,  Unit: nan  , OrderID 669249408.0, medication: sodium bicarbonate 8.4 % injection syringe
For given Med, Dose or Unit is NAN:  Dose: 50.0,  Unit: nan  , OrderID 669249407.0, medication: sodium bicarbonate 8.4 % injection syringe


In [61]:
for i in range(100):
    plt.close()

In [62]:
cohort

,mrn,datetime,Weight,Admission,Discharge
0,3337103,2020-04-08,95.4,2020-04-07 14:35:00,2020-05-08 23:59:00
1,5236378,2020-04-04,104.3,2020-04-04 00:24:00,2020-05-12 23:59:00


In [138]:
subject = 0

mrn = cohort.iloc[subject].mrn
meds_ts_df_mrn = meds_ts_df_mrn_all[subject]

In [139]:
meds_to_plot = [x for x in meds_ts_df_mrn.columns if 'propofol' in x]

In [140]:
for i in range(100):
    plt.close()

In [141]:
np.unique([x for x in df.MedicationDSC if 'propofol' in x])

array(['propofol (diprivan) infusion 10 mg/ml',
       'propofol 10 mg/ml intravenous emulsion',
       'propofol bolus from infusion 10 mg/ml (premix)'], dtype='<U46')

In [142]:
# med_to_plot = 'propofol bolus from infusion 10 mg/ml (premix)'
med_to_plot = 'propofol_10_mgperml_intravenous_emulsion'

for med_to_plot in meds_to_plot:
# 'propofol (diprivan) infusion 10 mg/ml',
       
#        'propofol bolus from infusion 10 mg/ml (premix)'], dtype='<U46')

# for med_to_plot in meds_ts_df_mrn.columns:
    med_name = med_to_plot.replace('_', ' ').replace('_','-').replace('perc', '%').replace('per', '/')
    erx = int(pd.unique(df[df.MedicationDSC.str.lower() == med_name].MedicationID)[0])
    
    plt.figure(figsize=(8,3))
    plt.plot(meds_ts_df_mrn[med_to_plot])
    plt.title('MRN: ' + str(mrn) + ', ' + med_name + ', erx=' + str(erx))
    plt.savefig('MRN_' + str(mrn) + '_' + med_to_plot + '_erx=' + str(erx)+'.jpg')
    plt.ylabel('mg (or mg/h)')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [128]:
propofol_variants = np.unique([x for x in edw_data.MedicationDSC if 'propofol' in x.lower()])
propofol_variants

array(['propofol (diprivan) infusion 10 mg/ml',
       'propofol 10 mg/ml intravenous emulsion',
       'propofol bolus from infusion 10 mg/ml (premix)',
       'propofol ivpb 10 mg/ml'], dtype='<U46')

In [132]:
subject = 1
mrn_sel = cohort.iloc[subject].mrn
# propofol_variant = propofol_variants[0]
for propofol_variant in propofol_variants:
    propofol_variant_name = propofol_variant.replace(' ', '_').replace('-','_').replace('%', 'perc').replace('/', 'per')
    df_sel = df.loc[(df.MRN == mrn_sel) & (df.MedicationDSC == propofol_variant)].sort_values(by = 'MedicationTakenDTS')
    print(df_sel.shape)
    if df_sel.shape[0] == 0:
        continue
    df_sel.to_csv(str(mrn_sel) + '_' + propofol_variant_name + '.csv', index=False)

(0, 40)
(27, 40)
(0, 40)
(0, 40)


In [ ]:
MARAction_color = {
    'New Bag' : 'blue',
    'Given': 'blue',
    'Restarted': 'cyan',
    'Same Bag': 'lightblue',
    'Rate Change' : 'red',
    'Rate Verify' : 'green',
    'Independent Double Check' : 'forestgreen',
    'Stopped' : 'black',
    'Canceled Entry': 'gray',
    'Missed': 'orange',
    'Held by provider':'brown',
    'MAR Hold':'brown',
    'Automatically Held':'brown',
    'Unheld by provider':'darkblue',
    'MAR Unhold':'darkblue',
    'Continued with Disposition from ED':'cyan',
    'See Alternative':'gray',
    'Bolus from Bag':'purple',
    'Due':'black',
    'Paused' : 'black',
    'Given by Other': 'blue',
     'Acknowledged':'gray',
     'Anesthesia Volume Adjustment':'gray',
     'Bolus':'purple',
     'Chemo Label RN Verify #1': 'gray',
     'Chemo Label RN Verify #2': 'gray',
     'Connected': 'gray',
     'Given During Downtime':'blue',
     'Held':'brown',
     'Infusion from Other Partners Entity':'gray',
     'Override Pull':'gray',
     'Override pull for Anesthesia':'gray',
     'Patch Applied':'gray',
     'Patch Removed':'gray',
     'Patch/Paste Applied':'gray',
     'Patch/Paste Removed':'gray',
     'Patient report of self administered meds':'gray',
     'Pending':'gray',
     'Refused': 'black',
     'Return to Cabinet': 'black',
     'Started During Downtime':'gray',
}

MARAction_int = {
    'New Bag' : 1,
    'Given':1,
    'Same Bag' : 1.2,
    'Restarted': 1.5,
    'Rate Change' : 3,
    'Rate Verify' : 2,
    'Independent Double Check' : 1.5,
    'Stopped' : 3.5,
    'Canceled Entry' : 4,
    'Missed': 4.5,
    'Held by provider': 3.8,
    'Automatically Held':3.8,
    'Unheld by provider':3.8,
    'Continued with Disposition from ED':1.5,
    'See Alternative':4,
    'Bolus from Bag':1.2,
    'Due':3.5,
    'MAR Unhold':1.5,
    'MAR Hold':3.8,
    'Paused':3.5,
    'Given by Other':1,
    'Acknowledged':5,
     'Anesthesia Volume Adjustment':5,
     'Bolus':1,
     'Chemo Label RN Verify #1': 5,
     'Chemo Label RN Verify #2': 5,
     'Connected': 3,
     'Given During Downtime':2,
     'Held':5,
     'Infusion from Other Partners Entity':5,
     'Override Pull':5,
     'Override pull for Anesthesia':5,
     'Patch Applied':5,
     'Patch Removed':5,
     'Patch/Paste Applied':5,
     'Patch/Paste Removed':5,
     'Patient report of self administered meds':5,
     'Pending':5,
     'Refused': 5,
     'Return to Cabinet': 5,
     'Started During Downtime':5,
}

def infusion_rate_plot_v1():

    fig, ax1 = plt.subplots(1,1, sharex=True, figsize=(8,4))
    ax1.plot(med_data.MedicationTakenDTS, med_data.InfusionRateNBR)
    ax1.tick_params(axis='x', rotation=90)

    ax1.set_title(med_name)
    ax1.set_ylabel(infusion_unit)

    ax1.plot(med_data.MedicationTakenDTS, med_data.MinimumDoseAMT, lw=1, c='gray',ls='--')
    ax1.plot(med_data.MedicationTakenDTS, med_data.MaximumDoseAMT, lw=1, c='gray')

    max_infusion = med_data.InfusionRateNBR.max()
    for MARAction_tmp in pd.unique(med_data.MARActionDSC):
        med_MARAction = med_data[med_data.MARActionDSC == MARAction_tmp]
        ax1.scatter(med_MARAction.MedicationTakenDTS, [max_infusion + MARAction_int[MARAction_tmp]]*med_MARAction.shape[0], color=MARAction_color[MARAction_tmp], s=5)

    ax1.tick_params(axis='x', rotation=90)

    legend = ['InfusionRateNBR'] + ['MinimumDoseAMT', 'MaximumDoseAMT'] + list(pd.unique(med_data.MARActionDSC)) 
    ax1.legend(legend, bbox_to_anchor=(1, -0.35), ncol=3)

    fig.subplots_adjust(bottom=0.4)
    plt.savefig(str(mrn)+'_'+med_name_short+'.png', dpi=400)
    
    plt.close()
# ax2.tick_params(axis='x', rotation=90)
